In [4]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv
from utils import combine_paragraphs, read_prompt_from_file_only

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
MONGO_COLLECTION = os.getenv("MONGO_COLLECTION_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, MONGO_COLLECTION]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]
    articles_collection = db[MONGO_COLLECTION] 

    client.admin.command('ping')
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

✅ Connected to MongoDB Atlas! Database: tuone


In [5]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
openAI_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openAI_api_key)

def ping_openai(client):
    try:
        response = client.models.list()
        print("✅ Successfully connected to OpenAI API!")
        print(f"Available Models: {[model.id for model in response.data]}")
    except Exception as e:
        print(f"❌ OpenAI API Connection Error: {e}")
ping_openai(client)

✅ Successfully connected to OpenAI API!
Available Models: ['gpt-4o-audio-preview-2024-12-17', 'dall-e-3', 'dall-e-2', 'gpt-4o-audio-preview-2024-10-01', 'text-embedding-3-small', 'gpt-4.1-nano', 'gpt-4.1-nano-2025-04-14', 'gpt-4o-realtime-preview-2024-10-01', 'gpt-4o-realtime-preview', 'babbage-002', 'gpt-4', 'text-embedding-ada-002', 'chatgpt-4o-latest', 'text-embedding-3-large', 'gpt-4o-mini-audio-preview', 'gpt-4o-audio-preview', 'o1-preview-2024-09-12', 'gpt-4.1-mini', 'gpt-3.5-turbo-instruct-0914', 'gpt-4o-mini-search-preview', 'gpt-4.1-mini-2025-04-14', 'davinci-002', 'gpt-3.5-turbo-1106', 'gpt-4o-search-preview', 'gpt-4-turbo', 'gpt-3.5-turbo-instruct', 'gpt-3.5-turbo', 'gpt-4-turbo-preview', 'gpt-4o-mini-search-preview-2025-03-11', 'gpt-4-0125-preview', 'gpt-4o-2024-11-20', 'whisper-1', 'gpt-4o-2024-05-13', 'gpt-4-turbo-2024-04-09', 'gpt-3.5-turbo-16k', 'gpt-image-1', 'o1-preview', 'gpt-4-0613', 'gpt-4.5-preview', 'gpt-4.5-preview-2025-02-27', 'gpt-4o-search-preview-2025-03-11'

In [6]:
# this function reconstructs the nodes into the format they were originally outputted

import json
from collections import defaultdict

def reconstruct_function_call_arguments(validated_nodes):
    grouped_nodes = defaultdict(list)

    for node in validated_nodes:
        node_type = node["type"]
        reconstructed_node = {
            "id": node.get("id"),
            "type": node_type,
            "name": node.get("name"),
        }

        # Include optional fields if present
        if node.get("location"):
            reconstructed_node["location"] = {
                "city": node["location"].get("city", ""),
                "country": node["location"].get("country", "")
            }

        if node.get("amount") is not None:
            reconstructed_node["amount"] = node["amount"]
        if node.get("status") is not None:
            reconstructed_node["status"] = node["status"]

        grouped_nodes[node_type].append(reconstructed_node)

    return json.dumps({"nodes": grouped_nodes}, ensure_ascii=False)

In [7]:
output_file = "nodes.jsonl"

PROMPT_PATH = "/Users/ben/Documents/bruegel/data_new/WORKING/INDUSTRY/tuone_v6/llmProcess/prompts/entities-only.txt"
system_prompt = read_prompt_from_file_only(PROMPT_PATH)

articles_to_process = list(
    articles_collection.find(
        {"validation": True},
        {"_id": 1, "paragraphs": 1, "nodes": 1, "validation": 1}
    )
    .sort("_id", -1)   
)

with open(output_file, "w", encoding="utf-8") as f_out:
    for article in articles_to_process:
        articleID = str(article["_id"])

        if article.get("validation") is not True:
            print(f"⏭️ Skipping Article ID: {articleID} - not validated")
            continue

        text = combine_paragraphs(article)
        user_content = f"Here is the article: {text}"

        nodes = article.get("nodes")
        if not nodes:
            print(f"⚠️ Article ID: {articleID} has no nodes — skipping")
            continue

        assistant_content = reconstruct_function_call_arguments(nodes)

        # Create fine-tuning format
        messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_content
            },
            {
                "role": "assistant",
                "content": assistant_content
            }
        ]

        f_out.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
        print(f"✅ Wrote example for Article ID: {articleID}")

print(f"\n🎉 Finished exporting fine-tuning examples to {output_file}")


✅ Wrote example for Article ID: 67fbf9b794b7f855cfea75d8
✅ Wrote example for Article ID: 67fbf9b694b7f855cfea75d7
✅ Wrote example for Article ID: 67fbf9b694b7f855cfea75d6
✅ Wrote example for Article ID: 67fbf9b694b7f855cfea75d5
✅ Wrote example for Article ID: 67fbf9b694b7f855cfea75d4
✅ Wrote example for Article ID: 67fbf9b594b7f855cfea75d3
✅ Wrote example for Article ID: 67fbf9b594b7f855cfea75d2
✅ Wrote example for Article ID: 67fbf9b594b7f855cfea75d1
✅ Wrote example for Article ID: 67fbf9b494b7f855cfea75cf
✅ Wrote example for Article ID: 67fbf9b494b7f855cfea75ce
✅ Wrote example for Article ID: 67fbf9b494b7f855cfea75cd
✅ Wrote example for Article ID: 67fbf9b494b7f855cfea75cc
✅ Wrote example for Article ID: 67fbf9b394b7f855cfea75cb
✅ Wrote example for Article ID: 67fbf9b394b7f855cfea75ca
✅ Wrote example for Article ID: 67fbf9b394b7f855cfea75c8
✅ Wrote example for Article ID: 67fbf9b394b7f855cfea75c7
✅ Wrote example for Article ID: 67fbf9b294b7f855cfea75c6
✅ Wrote example for Article ID: